<a href="https://colab.research.google.com/github/amjadsaadalqhtani-del/Smart_Event_Manager/blob/main/Day4_Activity2_Guardrails_Validation_PARTICIPANT_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **AGENTIC AI BOOTCAMP — Day 4 Activity 2 | Participant Version**  
> Find every `TODO` and fill in the blanks. Do **not** change anything outside TODO sections.

---

# Day 4 - Activity 2: Implementing Guardrails and Validation

---

## Overview

In Activity 1, you built a two-agent crew that produces and critiques research. The system is more reliable than a single agent, but it still has a critical problem for production use: the outputs are free-form text. There is no guarantee that the Critic's response will always contain the same fields, in the same format, with the same data types. Downstream systems — databases, dashboards, APIs — cannot reliably consume free-form text.

In this activity, you will solve that problem by adding two layers of safety:

1. **Structured output validation** — forcing the Critic to produce output that conforms to a strict JSON schema, and automatically rejecting responses that do not comply
2. **Human-in-the-loop oversight** — pausing the pipeline before finalization and requiring explicit human approval

These are not optional features for production systems — they are requirements. Any autonomous process that writes to a database, sends communications, or makes decisions needs both.

---

## Learning Goals

By the end of this activity, you will be able to:

- Define a JSON schema using Pydantic and explain why schema validation matters in production
- Configure a CrewAI agent to produce structured output conforming to a Pydantic model
- Write a validation function that catches and handles schema violations gracefully
- Implement a human-in-the-loop checkpoint using both a terminal prompt and a Gradio interface
- Articulate when and why human oversight is required in autonomous AI pipelines

---

## Conceptual Background: Why Guardrails?

Consider what happens when an autonomous agent pipeline runs without guardrails in a production environment:

```
WITHOUT GUARDRAILS:
  Agent produces: "The report looks mostly good, I give it a 7/10"
  Database expects: {"rating": "Good", "score": 7, "issues": [...]}
  Result: Database write fails silently, or stores malformed data

WITH SCHEMA VALIDATION:
  Agent produces: "The report looks mostly good, I give it a 7/10"
  Validator catches: rating field missing, score is not in allowed enum
  Result: Pipeline raises a controlled exception, logs the error,
          and either retries with corrected instructions or escalates to a human
```

### The Three Guardrail Layers Covered in This Activity

| Layer | What It Does | When It Fires |
|---|---|---|
| Pydantic schema | Defines the required structure of the output | At agent output definition time |
| Validation function | Checks the actual output against the schema | After every agent response |
| Human-in-the-loop | Pauses execution for human review and approval | Before any irreversible action (saving, sending, publishing) |

---

**Estimated Time:** 55-70 minutes  
**Difficulty:** Intermediate to Advanced  
**Tools Required:** CrewAI, Pydantic, Gradio, OpenAI API  
**Prerequisites:** Day 4 Activity 1 (two-agent CrewAI system)

---
## Step 0: Install Dependencies

In [2]:
!pip install -q -U crewai crewai-tools langchain-google-genai google-generativeai pydantic gradio

print("All packages installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.5/811.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.

---
## Step 1: Configure API Key

In [28]:
import os

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    # TODO: Replace with your actual API key if not using Colab secrets.
    os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY_HERE"  # استبدليه بمفتاحكِ الخاص
    print("WARNING: API key set manually. Do not share this notebook.")


---
## Step 2: Define the JSON Schema Using Pydantic

### What is Pydantic?

Pydantic is a Python library for data validation. You define the shape of your data as a class, and Pydantic enforces that any data assigned to that class matches the specified types, constraints, and required fields.

When used with LangChain and CrewAI, Pydantic models serve a second purpose: they are automatically converted into JSON Schema format and included in the prompt sent to the LLM, instructing it on exactly how to format its response.

### Schema Design Principles

When designing an output schema for an LLM agent, follow these rules:

1. **Use enums for categorical fields** — do not allow free-text where a fixed set of values is expected (e.g., a quality rating should be an enum, not a string)
2. **Use lists for variable-length collections** — issues, strengths, and recommendations are naturally variable in count
3. **Add field descriptions** — these descriptions are read by the LLM and significantly improve compliance
4. **Constrain string lengths where possible** — this prevents the agent from writing an essay in a field that should contain a summary sentence
5. **Make nothing optional unless it genuinely can be absent** — optional fields are often omitted by the LLM

In [5]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Literal
from enum import Enum


# ─── QualityRating Enum ───────────────────────────────────────────────────────
class QualityRating(str, Enum):
    POOR      = "Poor"
    FAIR      = "Fair"
    GOOD      = "Good"
    EXCELLENT = "Excellent"


# ─── CritiqueIssue ────────────────────────────────────────────────────────────
class CritiqueIssue(BaseModel):
    category: Literal["Factual Gap", "Logical Flaw", "Potential Hallucination", "Missing Perspective"] = Field(
        description="The type of issue identified."
    )
    description: str = Field(
        description="A specific explanation of the problem. Must reference the exact passage.",
        min_length=20, max_length=500
    )
    severity: Literal["Low", "Medium", "High"] = Field(
        description="How significantly this issue affects reliability."
    )


# ─── CritiqueOutput Schema ───────────────────────────────────────────────────
class CritiqueOutput(BaseModel):
    """
    Complete structured output the Critic Agent must produce.
    All fields are required — the agent's response is rejected if any field is missing.
    """
    topic: str = Field(
        description="The research topic that was evaluated."  # TODO 1a
    )

    strengths: List[str] = Field(
        description="A list of specific strengths identified in the briefing.",  # TODO 1b
        min_length=2  # TODO 1c — minimum 2 strengths
    )

    issues: List[CritiqueIssue] = Field(
        description="A list of all identified problems or issues."  # TODO 1d
    )

    overall_rating: QualityRating = Field(  # TODO 1e — Enum enforces the exact 4 values
        description="Overall quality rating: Poor, Fair, Good, or Excellent."
    )

    rating_justification: str = Field(
        description="A specific explanation justifying the overall quality rating assigned.",  # TODO 1f
        min_length=50, max_length=400
    )

    recommendation: str = Field(
        description="A single actionable improvement recommendation for the report.",  # TODO 1g
        min_length=30, max_length=300
    )

    ready_to_publish: bool = Field(
        description="True only if the briefing requires no major revision and is ready for publication."  # TODO 1h
    )

    # ─── Custom Validator ─────────────────────────────────────────────────────
    @field_validator("strengths")
    @classmethod
    def strengths_must_be_specific(cls, v):
        # TODO 2a — Set of blocked generic terms (converted to lowercase for accurate checking)
        generic_terms = {"clear", "good", "comprehensive", "well-written", "detailed"}

        for strength in v:
            # TODO 2b — Check if cleaned strength matches any blocked term
            if strength.strip().lower() in generic_terms:
                # TODO 2c — Descriptive error message
                raise ValueError(
                    f"Strength '{strength}' is too generic. Please provide a more specific, detailed strength."
                )
        return v


print('CritiqueOutput schema defined.')
print('Required fields:')
for name, info in CritiqueOutput.model_fields.items():
    print(f'  {name}: {info.annotation}')

CritiqueOutput schema defined.
Required fields:
  topic: <class 'str'>
  strengths: typing.List[str]
  issues: typing.List[__main__.CritiqueIssue]
  overall_rating: <enum 'QualityRating'>
  rating_justification: <class 'str'>
  recommendation: <class 'str'>
  ready_to_publish: <class 'bool'>


---
## Step 3: Define the Agents

The Researcher Agent is the same as in Activity 1. The Critic Agent is updated with an `output_pydantic` parameter that tells CrewAI to enforce the schema on its output.

When `output_pydantic` is set, CrewAI automatically:
1. Converts the Pydantic model to JSON Schema format
2. Includes the schema in the task prompt
3. Parses and validates the agent's response against the schema
4. Raises a structured error if validation fails

In [29]:
from crewai import Agent, Task, Crew, Process, LLM

# استخدام الموديل بدون أرقام
researcher_llm = LLM(model="gemini-1.5-flash", temperature=0.7)
critic_llm     = LLM(model="gemini-1.5-flash", temperature=0.0)

# أو يمكنك استخدام النسخة المتقدمة Pro بدون أرقام أيضاً:
# researcher_llm = LLM(model="gemini-pro", temperature=0.7)

# Researcher Agent
researcher_agent = Agent(
    role="Senior Research Analyst",
    goal=(
        "Produce a comprehensive, well-structured research briefing on complex topics. "
        "Cover background context, current state, key debates, real-world implications, "
        "and clearly labeled uncertainties."
    ),
    backstory=(
        "You are a senior research analyst with 15 years of experience producing "
        "thorough briefings for decision-makers. You never present speculation as fact, "
        "always acknowledge what is unknown, and structure your work for non-expert audiences "
        "without losing nuance."
    ),
    llm=researcher_llm,
    verbose=True,
    allow_delegation=False
)

# Critic Agent
critic_agent = Agent(
    role="Critique Expert",
    goal=(
        "Provide constructive criticism on research briefings. "
        "Evaluate for factual accuracy, logical consistency, potential biases, "
        "and clarity of presentation. Your output MUST conform to the specified Pydantic schema." # TODO 3a - explicit Pydantic instruction
    ),
    backstory=(
        "You are a meticulous critique expert with a keen eye for detail. "
        "You pride yourself on identifying subtle flaws, ensuring objectivity, "
        "and helping improve the quality of research. You always provide actionable feedback." # TODO 3b - explicit actionable feedback instruction
    ),
    llm=critic_llm,
    verbose=True,
    allow_delegation=False
)


---
## Step 4: Define Tasks with Schema Enforcement

The key change from Activity 1 is the `output_pydantic` parameter on the critique task. This is what triggers schema validation. The task description is also updated to explicitly instruct the agent on the required output format.

In [43]:
import json

RESEARCH_TOPIC = "The potential risks and benefits of artificial general intelligence (AGI)"

# Task 1: Research Briefing — complete, do not modify.
research_task = Task(
    description=(
        f"Produce a comprehensive research briefing on the following topic:\n"
        f"Topic: {RESEARCH_TOPIC}\n\n"
        "Structure the briefing with these five sections:\n"
        "1. Background\n2. Current State\n3. Key Debates\n4. Implications\n5. Uncertainties\n\n"
        "Each section must be at least two substantial paragraphs. "
        "Distinguish clearly between established facts and speculation. "
        "Do not fabricate specific statistics or citations."
    ),
    expected_output=(
        "A structured five-section research briefing, 600-900 words total, "
        "written as a professional document suitable for a senior audience."
    ),
    agent=researcher_agent
)


# ─── Task 2: Critique Task ───────────────────────────────────────────────────
critique_task = Task(
    description=(
        f"Critique the research briefing produced for the topic: '{RESEARCH_TOPIC}'.\n"
        "Evaluate the briefing for factual gaps, logical flaws, potential hallucinations, and missing perspectives.\n\n"
        "Instructions:\n"
        "- Identify at least 2 specific strengths (do NOT use generic terms like 'clear', 'good', or 'detailed').\n"
        "- Detail all identified issues with accurate categories, descriptions, and severity levels.\n"
        "- Assign an overall quality rating (Poor, Fair, Good, or Excellent) with a clear justification (50-400 chars).\n"
        "- Provide a single actionable recommendation (30-300 chars).\n"
        "- Set 'ready_to_publish' to True ONLY if no major revisions are needed."
    ),
    expected_output=(
        "A structured CritiqueOutput object containing a detailed evaluation of the research briefing."
    ),
    output_pydantic=CritiqueOutput,  # ينشئ الـ Schema ويتحقق من مخرجات Gemini تلقائياً
    agent=critic_agent,
    context=[research_task]
)

print('Tasks defined with schema enforcement.')
print(f'Topic: {RESEARCH_TOPIC}')

Tasks defined with schema enforcement.
Topic: The potential risks and benefits of artificial general intelligence (AGI)


---
## Step 5: Write the Validation Function

Even with `output_pydantic` set, it is good practice to write an explicit validation function. This gives you full control over what happens when validation fails — you can log the error, retry with modified instructions, alert a human operator, or raise an exception that halts the pipeline with a meaningful message.

Never rely silently on implicit validation in production code. Make failures loud and traceable.

In [10]:
import json
from typing import Union
from pydantic import ValidationError


# ─── TODO 4 Solution ─────────────────────────────────────────────────────────
def validate_critique_output(
    raw_output: Union[str, dict, CritiqueOutput]
) -> tuple[bool, Union[CritiqueOutput, list]]:
    """
    Validates the Critic Agent's output against the CritiqueOutput schema.
    Returns (True, CritiqueOutput) on success, (False, errors) on failure.
    """

    # Case 1: Already a validated Pydantic object
    if isinstance(raw_output, CritiqueOutput):
        print('Validation passed: already a CritiqueOutput object.')
        return True, raw_output

    # Case 2: Dict
    if isinstance(raw_output, dict):
        try:
            validated = CritiqueOutput.model_validate(raw_output)  # TODO 4a — validate dict against schema
            print('Validation passed: dict parsed into CritiqueOutput.')
            return True, validated
        except ValidationError as e:
            print(f'Validation FAILED: {len(e.errors())} violation(s).')
            return False, e.errors()

    # Case 3: String
    if isinstance(raw_output, str):
        cleaned = raw_output.strip()

        # TODO 4b — strip markdown code fences if present
        if cleaned.startswith("```json"):  # TODO 4b-i
            cleaned = cleaned[7:]          # TODO 4b-ii — remove ```json prefix
        if cleaned.startswith('```'):
            cleaned = cleaned[3:]
        if cleaned.endswith('```'):
            cleaned = cleaned[:-3]
        cleaned = cleaned.strip()

        try:
            data = json.loads(cleaned)     # TODO 4c — parse cleaned string as JSON
            validated = CritiqueOutput.model_validate(data)
            print('Validation passed: string JSON parsed into CritiqueOutput.')
            return True, validated
        except json.JSONDecodeError as e:
            print(f'Validation FAILED: JSON parse error: {e}')
            return False, [{"type": "json_parse_error", "msg": str(e)}]
        except ValidationError as e:
            print(f'Validation FAILED: {len(e.errors())} violation(s).')
            for err in e.errors():
                print(f"  Field: {' -> '.join(str(x) for x in err['loc'])}")
                print(f"  Error: {err['msg']}")
            return False, e.errors()

    return False, [{"type": "unexpected_type", "msg": str(type(raw_output))}]


def run_validation_with_retry(raw_output, max_retries: int = 1):
    is_valid, result = validate_critique_output(raw_output)
    if is_valid:
        return True, result
    print('Validation failed. In production, this triggers a retry with errors in the prompt.')
    for err in result:
        print(f'  - {err}')
    return False, None


print('Validation functions defined.')


Validation functions defined.


---
## Step 6: Run the Crew and Validate the Output

In [42]:
import nest_asyncio
import asyncio
nest_asyncio.apply()

research_crew = Crew(
    agents=[researcher_agent, critic_agent],
    tasks=[research_task, critique_task],
    process=Process.sequential,
    verbose=True
)

print("Starting crew execution...")
print("=" * 70)

async def run_crew_async():
    return  research_crew.kickoff_async()

crew_result = asyncio.run(run_crew_async())

print("=" * 70)
print("Crew execution complete. Running validation...")

Starting crew execution...
Crew execution complete. Running validation...


In [31]:
print(f"Researcher Agent LLM Model: {researcher_agent.llm.model}")
print(f"Critic Agent LLM Model: {critic_agent.llm.model}")

Researcher Agent LLM Model: gemini-1.5-flash
Critic Agent LLM Model: gemini-1.5-flash


In [32]:
# Extract the raw critique output
# CrewAI may return the Pydantic object directly if output_pydantic is set
raw_critique = critique_task.output

# Handle different possible return types from CrewAI versions
if hasattr(raw_critique, 'pydantic') and raw_critique.pydantic is not None:
    critique_data = raw_critique.pydantic
elif hasattr(raw_critique, 'raw'):
    critique_data = raw_critique.raw
else:
    critique_data = raw_critique

print("Raw output type:", type(critique_data))
print()

# Run validation
is_valid, validated_critique = run_validation_with_retry(critique_data)

if is_valid:
    print("\nValidation successful. Proceeding to human review step.")
else:
    print("\nValidation failed. Pipeline halted.")
    print("In a production system, this would trigger an alert and log the failure.")

Raw output type: <class 'NoneType'>

Validation failed. In production, this triggers a retry with errors in the prompt.
  - {'type': 'unexpected_type', 'msg': "<class 'NoneType'>"}

Validation failed. Pipeline halted.
In a production system, this would trigger an alert and log the failure.


---
## Step 7: Display the Validated Structured Output

Now that we have a validated Pydantic object, every field is guaranteed to be present and correctly typed. This is what gets passed to downstream systems.

In [41]:
from IPython.display import Markdown, display

if validated_critique is None:
    print("No validated output available. Check validation errors above.")
else:
    c = validated_critique  # Shorthand

    print("=" * 70)
    print("VALIDATED CRITIQUE OUTPUT")
    print("=" * 70)

    report_md = f"""
## Critical Review: {c.topic}

### Overall Rating: {c.overall_rating.value}

{c.rating_justification}

**Ready to Publish:** {'Yes' if c.ready_to_publish else 'No — Revision Required'}

---

### Strengths

{''.join(f'- {s}' + chr(10) for s in c.strengths)}

---

### Issues Identified ({len(c.issues)} total)

{''.join(f'**[{issue.severity} Severity] {issue.category}**' + chr(10) + issue.description + chr(10) + chr(10) for issue in c.issues)}

---

### Primary Recommendation

{c.recommendation}
"""

    display(Markdown(report_md))

    print("\nStructured field access (for downstream systems):")
    print("-" * 40)
    print(f"  topic:              {c.topic[:60]}...")
    print(f"  overall_rating:     {c.overall_rating.value}")
    print(f"  ready_to_publish:   {c.ready_to_publish}")
    print(f"  strengths count:    {len(c.strengths)}")
    print(f"  issues count:       {len(c.issues)}")
    high_issues = [i for i in c.issues if i.severity == 'High']
    print(f"  high severity:      {len(high_issues)}")

No validated output available. Check validation errors above.


---
## Step 8: Human-in-the-Loop Checkpoint — Terminal Version

### Why Human-in-the-Loop Matters

Schema validation ensures the output is correctly formatted. It does not ensure the output is correct, fair, or appropriate for the specific context. Those judgments require human intelligence.

A human-in-the-loop checkpoint pauses the pipeline before any irreversible action — writing to a database, sending an email, publishing content — and presents the output to a human for review. The human can approve (continue), reject (halt), or request revision (loop back).

This pattern is essential whenever:
- The output will be seen by an external audience
- The output will trigger a downstream action with real-world consequences
- The stakes of an error are high (financial, legal, reputational)
- The system is new and not yet trusted to operate fully autonomously

The terminal version below is the simplest implementation. It uses Python's built-in `input()` to pause execution and wait for a human response.

In [34]:
def human_approval_checkpoint_terminal(
    critique: CritiqueOutput,
    action_description: str = "save the critique to the database"
) -> tuple[bool, str]:

    print()
    print('=' * 70)
    print('HUMAN-IN-THE-LOOP REVIEW CHECKPOINT')
    print('=' * 70)
    print(f'Pending action: {action_description}')
    print('-' * 70)

    # TODO 5a — طباعة تفاصيل النقد الأساسية
    rating_val = critique.overall_rating.value if hasattr(critique.overall_rating, 'value') else critique.overall_rating
    print(f'Topic: {critique.topic}')
    print(f'Overall Rating: {rating_val}')
    print(f'Ready to Publish: {critique.ready_to_publish}')
    print(f'Issues Identified: {len(critique.issues)}')
    print(f'Justification: {critique.rating_justification}')
    print(f'Recommendation: {critique.recommendation}')

    print('-' * 70)
    print('Options: [A] Approve   [R] Reject   [V] Request Revision')
    print()

    while True:
        decision = input('Your decision (A/R/V): ').strip().upper()

        if decision == 'A':
            # TODO 5b — خيار القبول (Approve)
            print('\n[APPROVED] Action confirmed by human reviewer.')
            return True, 'approved'

        elif decision == 'R':
            # TODO 5c — خيار الرفض (Reject)
            reason = input('Optional reason for rejection: ').strip()
            print(f'\n[REJECTED] Critique rejected. Reason: {reason if reason else "None provided"}')
            return False, 'rejected'

        elif decision == 'V':
            # TODO 5d — خيار طلب التعديل (Revision)
            feedback = input('Revision feedback: ').strip()
            print(f'\n[REVISION REQUESTED] Feedback: {feedback}')
            return False, 'revision'

        else:
            print('Invalid input. Please enter A, R, or V.')


print('Terminal HITL function defined.')


Terminal HITL function defined.


In [35]:
# Run the human-in-the-loop checkpoint
# The script will pause and wait for your input

if validated_critique is not None:
    approved, decision = human_approval_checkpoint_terminal(
        critique=validated_critique,
        action_description="write the structured critique to the research database"
    )

    print()
    if approved:
        print("Human approved. Executing downstream action...")
        # In a real system, this is where you would call:
        # - write_to_database(validated_critique)
        # - send_notification_email(validated_critique)
        # - publish_to_cms(validated_critique)
        print("[Placeholder] Downstream action completed successfully.")
    else:
        print(f"Human decision: {decision}. No action taken.")
else:
    print("Skipping approval checkpoint: no validated output available.")

Skipping approval checkpoint: no validated output available.


---
## Step 9: Human-in-the-Loop Checkpoint — Gradio Interface

The terminal `input()` approach works in a notebook but is not suitable for production web applications or team use. Gradio provides a clean browser-based approval interface that can be shared with non-technical reviewers.

### Design of the Gradio Approval Interface

The interface presents the full critique in a readable format and provides three action buttons. The reviewer's decision and any feedback are captured and logged. The interface uses a shared state object to communicate the decision back to the main pipeline code.

In [36]:
import gradio as gr
import threading


def build_review_display(critique: CritiqueOutput) -> str:
    """Formats the validated critique as Markdown for the Gradio interface."""
    issues_md = ""
    for i, issue in enumerate(critique.issues, 1):
        issues_md += f"\n**Issue {i} [{issue.severity}] — {issue.category}**\n{issue.description}\n"
    strengths_md = "\n".join(f"- {s}" for s in critique.strengths)

    rating_val = critique.overall_rating.value if hasattr(critique.overall_rating, 'value') else critique.overall_rating

    return f"""
## Review Summary
**Topic:** {critique.topic}
**Rating:** {rating_val}
**Ready to Publish:** {'Yes' if critique.ready_to_publish else 'No'}

---
### Justification
{critique.rating_justification}

---
### Strengths
{strengths_md}

---
### Issues ({len(critique.issues)} total)
{issues_md}

---
### Recommendation
{critique.recommendation}
"""


def launch_gradio_approval(critique: CritiqueOutput) -> tuple[bool, str]:
    """
    Launches a Gradio interface for human review.
    Returns (approved: bool, decision: str).
    """
    review_state = {"decision": None, "feedback": None, "complete": False}
    stop_event = threading.Event()
    review_text = build_review_display(critique)

    # ─── TODO 6 Solutions ────────────────────────────────────────────────────
    def on_approve(feedback_text):
        # TODO 6a — Approval doesn't strictly require feedback
        review_state['decision'] = 'approved'
        review_state['feedback'] = feedback_text.strip() if feedback_text else None
        review_state['complete'] = True
        stop_event.set()
        return "Decision recorded: APPROVED. Closing interface..."

    def on_reject(feedback_text):
        # TODO 6b — Require non-empty feedback for rejection
        if not feedback_text or not feedback_text.strip():
            return "ERROR: Feedback is required to Reject."
        review_state['decision'] = 'rejected'
        review_state['feedback'] = feedback_text.strip()
        review_state['complete'] = True
        stop_event.set()
        return "Decision recorded: REJECTED. Closing interface..."

    def on_revise(feedback_text):
        # TODO 6c — Require non-empty feedback for revision
        if not feedback_text or not feedback_text.strip():
            return "ERROR: Feedback is required to Request Revision."
        review_state['decision'] = 'revision'
        review_state['feedback'] = feedback_text.strip()
        review_state['complete'] = True
        stop_event.set()
        return "Decision recorded: REVISION REQUESTED. Closing interface..."

    with gr.Blocks(theme=gr.themes.Default(), title="Human Review Checkpoint") as demo:
        gr.Markdown("# Human Review Checkpoint")
        gr.Markdown("Review the critique below and record your decision.")
        with gr.Row():
            with gr.Column(scale=2):
                gr.Markdown(review_text)
            with gr.Column(scale=1):
                gr.Markdown("### Your Decision")
                feedback_box = gr.Textbox(
                    label="Feedback (required for Reject and Revise)",
                    placeholder="Enter your feedback here...",
                    lines=5
                )
                approve_btn = gr.Button("Approve",          variant="primary")
                reject_btn  = gr.Button("Reject",           variant="stop")
                revise_btn  = gr.Button("Request Revision", variant="secondary")
                status_box  = gr.Textbox(label="Status", interactive=False,
                                         value="Awaiting your decision...")

                approve_btn.click(fn=on_approve, inputs=[feedback_box], outputs=[status_box])
                reject_btn.click( fn=on_reject,  inputs=[feedback_box], outputs=[status_box])
                revise_btn.click( fn=on_revise,  inputs=[feedback_box], outputs=[status_box])

    demo.launch(share=True, quiet=True)
    print("Waiting for reviewer decision...")
    stop_event.wait(timeout=600)
    demo.close()

    if not review_state["complete"]:
        print("Timed out after 10 minutes.")
        return False, "timeout"

    approved = (review_state["decision"] == "approved")
    print(f"Decision: {review_state['decision'].upper()}")
    if review_state["feedback"]:
        print(f"Feedback: {review_state['feedback']}")
    return approved, review_state["decision"]


print("Gradio approval interface defined.")


Gradio approval interface defined.


In [37]:
# Launch the Gradio approval interface
# A public URL will be printed below. Open it in your browser to review and approve.

if validated_critique is not None:
    print("Launching Gradio review interface...")
    print("Open the URL below to complete the human review step.")
    print()

    approved, decision = launch_gradio_approval(validated_critique)

    print()
    if approved:
        print("Human approved via Gradio interface.")
        print("Executing downstream action...")
        # Downstream action placeholder
        print("[Placeholder] Downstream action completed.")
    else:
        print(f"Human decision: {decision}. No downstream action taken.")
else:
    print("Skipping Gradio checkpoint: no validated output available.")

Skipping Gradio checkpoint: no validated output available.


---
## Step 10: Test Schema Validation Against Invalid Inputs

Understanding validation means understanding how it fails. Run these tests to see exactly what error messages your validation function produces when given malformed input. This is critical knowledge for debugging production pipelines.

In [38]:
print("VALIDATION FAILURE TESTS")
print("=" * 70)

# Test 1: Missing required field
print("\nTest 1: Missing required field (recommendation is absent)")
print("-" * 50)
missing_field = {
    "topic": "Test topic",
    "strengths": ["The briefing correctly identifies the scope of the problem", "Sources are appropriately hedged"],
    "issues": [{"category": "Factual Gap", "description": "The claim about market size lacks a cited source and a specific year", "severity": "Medium"}],
    "overall_rating": "Good",
    "rating_justification": "The briefing covers the main points but lacks citation discipline in several places.",
    # recommendation is intentionally missing
    "ready_to_publish": False
}
is_valid, result = validate_critique_output(missing_field)
print(f"Result: valid={is_valid}")


# Test 2: Invalid enum value
print("\nTest 2: Invalid enum value (rating is 'Very Good' not in allowed values)")
print("-" * 50)
invalid_enum = {
    "topic": "Test topic",
    "strengths": ["The briefing correctly identifies the scope of the problem", "Sources are appropriately hedged"],
    "issues": [{"category": "Factual Gap", "description": "The claim about market size lacks a cited source and a specific year", "severity": "Medium"}],
    "overall_rating": "Very Good",  # Not in the enum: Poor / Fair / Good / Excellent
    "rating_justification": "The briefing covers the main points but lacks citation discipline.",
    "recommendation": "Add specific citations for all statistical claims in the Current State section.",
    "ready_to_publish": False
}
is_valid, result = validate_critique_output(invalid_enum)
print(f"Result: valid={is_valid}")


# Test 3: Generic strength caught by custom validator
print("\nTest 3: Generic strength rejected by custom validator")
print("-" * 50)
generic_strength = {
    "topic": "Test topic",
    "strengths": ["clear", "comprehensive"],  # Generic terms blocked by validator
    "issues": [{"category": "Factual Gap", "description": "The claim about market size lacks a cited source and a specific year", "severity": "Low"}],
    "overall_rating": "Good",
    "rating_justification": "The briefing covers the main points but lacks citation discipline.",
    "recommendation": "Add specific citations for all statistical claims in the Current State section.",
    "ready_to_publish": True
}
is_valid, result = validate_critique_output(generic_strength)
print(f"Result: valid={is_valid}")


# Test 4: Valid input — should pass all checks
print("\nTest 4: Fully valid input — should pass all checks")
print("-" * 50)
valid_input = {
    "topic": "The potential risks and benefits of AGI",
    "strengths": [
        "The briefing correctly distinguishes between narrow AI and AGI in the Background section",
        "The Uncertainties section honestly acknowledges the lack of expert consensus on timelines"
    ],
    "issues": [
        {
            "category": "Potential Hallucination",
            "description": "The claim that '60% of AI researchers expect AGI by 2040' appears in the Key Debates section without a cited survey source",
            "severity": "High"
        },
        {
            "category": "Missing Perspective",
            "description": "The Implications section focuses on economic impacts but does not address geopolitical risks or international governance frameworks",
            "severity": "Medium"
        }
    ],
    "overall_rating": "Good",
    "rating_justification": "The briefing is well-structured and covers the major viewpoints, but the unsourced survey statistic is a significant reliability concern that must be addressed before publication.",
    "recommendation": "Replace the unsourced 60% statistic in the Key Debates section with a properly cited source, or remove it and rephrase as a general observation about expert disagreement.",
    "ready_to_publish": False
}
is_valid, result = validate_critique_output(valid_input)
print(f"Result: valid={is_valid}")

VALIDATION FAILURE TESTS

Test 1: Missing required field (recommendation is absent)
--------------------------------------------------
Validation FAILED: 1 violation(s).
Result: valid=False

Test 2: Invalid enum value (rating is 'Very Good' not in allowed values)
--------------------------------------------------
Validation FAILED: 1 violation(s).
Result: valid=False

Test 3: Generic strength rejected by custom validator
--------------------------------------------------
Validation FAILED: 1 violation(s).
Result: valid=False

Test 4: Fully valid input — should pass all checks
--------------------------------------------------
Validation passed: dict parsed into CritiqueOutput.
Result: valid=True


---
## Exercises and Challenges

### Level 1 - Foundational

1. Add a new required field to the `CritiqueOutput` schema called `word_count_estimate` (an integer representing the approximate length of the original briefing). Re-run the crew and verify the field is populated correctly.
2. Add a second custom validator that rejects any `rating_justification` shorter than 80 characters. Test it with a short justification and confirm the error is caught.
3. Modify the Gradio interface to display the full research briefing (not just the critique) in a third panel so the reviewer can read both documents side by side before making their decision.

### Level 2 - Intermediate

4. Implement automatic retry logic: if the Critic's output fails validation, automatically re-invoke the Critic with the specific validation error messages appended to its task description. Allow up to two retries before halting.
5. Add a logging step that writes every human decision (approval, rejection, or revision request) along with the timestamp, the rating, and the feedback text to a JSON log file. This creates an audit trail for the pipeline.
6. Modify the `CritiqueOutput` schema to include an `issues_summary` field — a plain-English one-sentence summary of all issues suitable for display in a notification email. Add a validator that checks it does not exceed 200 characters.

### Level 3 - Advanced

7. Connect the human approval checkpoint to the Google Sheets integration from Day 3 Activity 1. On approval, write the structured critique (all fields) to a new row in the sheet. On rejection, write the topic and rejection reason to a separate "Rejected" sheet.
8. Implement a full revision loop: on a "Revision Requested" decision, feed the human's feedback text back into the Researcher Agent as an additional constraint, re-run both agents, re-validate the output, and re-present the updated critique for human approval. Cap the loop at three revision cycles.
9. Build a complete dashboard in Gradio that shows: (a) the research briefing, (b) the structured critique with all fields displayed in a table format, (c) the human review panel, and (d) a history log showing all decisions made in the current session.

---

## Key Takeaways

| Concept | Why It Matters |
|---|---|
| Pydantic schema enforcement | Makes agent outputs machine-readable and reliably structured. Free-form text cannot be safely consumed by databases or downstream APIs |
| Enum fields for categorical values | Prevents the LLM from inventing its own categories. Without enums, a quality rating might come back as "7/10", "mostly good", "B+", or "Good" — all different strings meaning roughly the same thing |
| Custom validators | Domain-specific rules that Pydantic's built-in types cannot express. Essential for enforcing quality standards beyond type checking |
| Explicit validation functions | Even with output_pydantic set, writing your own validation function gives you control over failure behavior. Implicit validation fails silently; explicit validation fails loudly with useful diagnostics |
| Human-in-the-loop before irreversible actions | The correct mental model is: agents are fast and scalable; humans are slow and irreplaceable. Use agents for everything that can be automated; require humans for everything that cannot be undone |
| Audit trail | Every human decision in an autonomous pipeline must be logged with timestamp, context, and outcome. Without this, there is no accountability and no basis for improving the system |

---

Excellent work. You have now built a production-grade multi-agent pipeline with structured output validation and human oversight. This completes Day 4.